[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-4/parallelization.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58239934-lesson-1-parallelization)

# 并行节点执行

## 回顾

在模块 3 中，我们深入探讨了"人机协同"（`human-in-the loop`），展示了 3 个常见用例：

(1) `Approval`（审批）—— 我们可以中断 Agent，将状态展示给用户，并允许用户批准某个操作

(2) `Debugging`（调试）—— 我们可以回退图的执行以重现或避免问题

(3) `Editing`（编辑）—— 你可以修改状态

## 目标

本模块将基于"人机协同"以及模块 2 中讨论的`记忆`概念。

我们将深入探讨`多 Agent`工作流，并构建一个多 Agent 研究助手，将本课程的所有模块内容串联起来。

为了构建这个多 Agent 研究助手，我们首先讨论一些 LangGraph 可控性主题。

我们将从[并行化](https://docs.langchain.com/oss/python/langgraph/how-tos/graph-api#create-branches)开始。

## 扇出与扇入

让我们构建一个简单的线性图，在每一步覆盖状态。

In [ ]:
%%capture --no-stderr
%pip install -U  langgraph langchain-tavily wikipedia langchain_deepseek langchain_community langgraph_sdk

In [ ]:
import os, getpass
from dotenv import load_dotenv

load_dotenv()

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("DEEPSEEK_API_KEY")

In [ ]:
from IPython.display import Image, display

from typing import Any, List
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    # 注意，没有 reducer 函数
    state: List[str]

class ReturnNodeValue:
    def __init__(self, node_secret: str):
        self._value = node_secret

    def __call__(self, state: State) -> Any:
        print(f"Adding {self._value} to {state['state']}")
        return {"state": [self._value]}

# 添加节点
builder = StateGraph(State)

# 使用 node_secret 初始化每个节点
builder.add_node("a", ReturnNodeValue("I'm A"))
builder.add_node("b", ReturnNodeValue("I'm B"))
builder.add_node("c", ReturnNodeValue("I'm C"))
builder.add_node("d", ReturnNodeValue("I'm D"))

# 流程
builder.add_edge(START, "a")
builder.add_edge("a", "b")
builder.add_edge("b", "c")
builder.add_edge("c", "d")
builder.add_edge("d", END)
graph = builder.compile()

display(Image(graph.get_graph().draw_mermaid_png()))

我们如预期的那样覆盖了状态。

In [ ]:
graph.invoke({"state": []})

现在，让我们并行运行 `b` 和 `c`。

然后运行 `d`。

我们可以通过从 `a` 扇出到 `b` 和 `c`，然后扇入到 `d` 来轻松实现。

状态更新在每个步骤结束时应用。

让我们运行它。

In [ ]:
builder = StateGraph(State)

# 使用 node_secret 初始化每个节点
builder.add_node("a", ReturnNodeValue("I'm A"))
builder.add_node("b", ReturnNodeValue("I'm B"))
builder.add_node("c", ReturnNodeValue("I'm C"))
builder.add_node("d", ReturnNodeValue("I'm D"))

# 流程
builder.add_edge(START, "a")
builder.add_edge("a", "b")
builder.add_edge("a", "c")
builder.add_edge("b", "d")
builder.add_edge("c", "d")
builder.add_edge("d", END)
graph = builder.compile()

display(Image(graph.get_graph().draw_mermaid_png()))

**我们看到一个错误**！

这是因为 `b` 和 `c` 在同一个步骤中写入了相同的状态键/通道。

In [ ]:
from langgraph.errors import InvalidUpdateError
try:
    graph.invoke({"state": []})
except InvalidUpdateError as e:
    print(f"An error occurred: {e}")

当使用扇出时，如果多个步骤正在写入同一个通道/键，我们需要确保使用了 reducer。

正如我们在模块 2 中提到的，`operator.add` 是 Python 内置 operator 模块的一个函数。

当 `operator.add` 应用于列表时，它执行列表拼接。

In [ ]:
import operator
from typing import Annotated

class State(TypedDict):
    # operator.add reducer 函数使此字段变为仅追加模式
    state: Annotated[list, operator.add]

# 添加节点
builder = StateGraph(State)

# 使用 node_secret 初始化每个节点
builder.add_node("a", ReturnNodeValue("I'm A"))
builder.add_node("b", ReturnNodeValue("I'm B"))
builder.add_node("c", ReturnNodeValue("I'm C"))
builder.add_node("d", ReturnNodeValue("I'm D"))

# 流程
builder.add_edge(START, "a")
builder.add_edge("a", "b")
builder.add_edge("a", "c")
builder.add_edge("b", "d")
builder.add_edge("c", "d")
builder.add_edge("d", END)
graph = builder.compile()

display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
graph.invoke({"state": []})

现在我们看到 `b` 和 `c` 并行执行的更新被追加到了状态中。

## 等待节点完成

现在，让我们考虑一个并行路径比另一条路径有更多步骤的情况。

In [ ]:
builder = StateGraph(State)

# 使用 node_secret 初始化每个节点
builder.add_node("a", ReturnNodeValue("I'm A"))
builder.add_node("b", ReturnNodeValue("I'm B"))
builder.add_node("b2", ReturnNodeValue("I'm B2"))
builder.add_node("c", ReturnNodeValue("I'm C"))
builder.add_node("d", ReturnNodeValue("I'm D"))

# 流程
builder.add_edge(START, "a")
builder.add_edge("a", "b")
builder.add_edge("a", "c")
builder.add_edge("b", "b2")
builder.add_edge(["b2", "c"], "d")
builder.add_edge("d", END)
graph = builder.compile()

display(Image(graph.get_graph().draw_mermaid_png()))

在这种情况下，`b`、`b2` 和 `c` 都属于同一个步骤。

图将等待所有这些节点完成，然后再进入步骤 `d`。

In [ ]:
graph.invoke({"state": []})

## 设置状态更新的顺序

然而，在每个步骤中，我们无法具体控制状态更新的顺序！

简单来说，这是由 LangGraph 根据图拓扑决定的确定性顺序，**我们无法控制**。

上面，我们看到 `c` 在 `b2` 之前添加。

但是，我们可以使用自定义 reducer 来自定义此顺序，例如对状态更新进行排序。

In [ ]:
def sorting_reducer(left, right):
    """ 合并并排序列表中的值"""
    if not isinstance(left, list):
        left = [left]

    if not isinstance(right, list):
        right = [right]
    
    return sorted(left + right, reverse=False)

class State(TypedDict):
    # sorting_reducer 将对状态中的值进行排序
    state: Annotated[list, sorting_reducer]

# 添加节点
builder = StateGraph(State)

# 使用 node_secret 初始化每个节点
builder.add_node("a", ReturnNodeValue("I'm A"))
builder.add_node("b", ReturnNodeValue("I'm B"))
builder.add_node("b2", ReturnNodeValue("I'm B2"))
builder.add_node("c", ReturnNodeValue("I'm C"))
builder.add_node("d", ReturnNodeValue("I'm D"))

# 流程
builder.add_edge(START, "a")
builder.add_edge("a", "b")
builder.add_edge("a", "c")
builder.add_edge("b", "b2")
builder.add_edge(["b2", "c"], "d")
builder.add_edge("d", END)
graph = builder.compile()

display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
graph.invoke({"state": []})

现在，reducer 对更新后的状态值进行了排序！

`sorting_reducer` 示例对所有值进行全局排序。我们还可以：

1. 在并行步骤中将输出写入状态中的单独字段
2. 在并行步骤之后使用"汇聚"（sink）节点来合并和排序这些输出
3. 合并后清除临时字段

<!-- 有关更多详细信息，请参见 [docs](https://docs.langchain.com/oss/python/langgraph/how-tos/graph-api#create-branches)。-->

## 与 LLM 协作

现在，让我们添加一个实际的例子！

我们想从两个外部来源（Wikipedia 和网络搜索）收集上下文，并让 LLM 回答一个问题。

In [ ]:
from langchain_deepseek import ChatDeepSeek
llm = ChatDeepSeek(model="deepseek-chat", temperature=0) 

In [ ]:
class State(TypedDict):
    question: str
    answer: str
    context: Annotated[list, operator.add]

你可以尝试不同的网络搜索工具。[Tavily](https://tavily.com/) 是一个不错的选择，但请确保你的 `TAVILY_API_KEY` 已设置。

In [ ]:
import os, getpass
def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")
_set_env("TAVILY_API_KEY")

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

from langchain_community.document_loaders import WikipediaLoader
from langchain_tavily import TavilySearch  # 自视频录制以来已更新

def search_web(state):
    
    """ 从网络搜索中检索文档 """

    # 搜索
    tavily_search = TavilySearch(max_results=3)
    data = tavily_search.invoke({"query": state['question']})
    search_docs = data.get("results", data)

     # 格式化
    formatted_search_docs = "\n\n---\n\n".join(
        [
            f'<Document href="{doc["url"]}">\n{doc["content"]}\n</Document>'
            for doc in search_docs
        ]
    )

    return {"context": [formatted_search_docs]} 

def search_wikipedia(state):
    
    """ 从 Wikipedia 检索文档 """

    # 搜索
    search_docs = WikipediaLoader(query=state['question'], 
                                  load_max_docs=2).load()

     # 格式化
    formatted_search_docs = "\n\n---\n\n".join(
        [
            f'<Document source="{doc.metadata["source"]}" page="{doc.metadata.get("page", "")}">\n{doc.page_content}\n</Document>'
            for doc in search_docs
        ]
    )

    return {"context": [formatted_search_docs]} 

def generate_answer(state):
    
    """ 用于回答问题的节点 """

    # 获取状态
    context = state["context"]
    question = state["question"]

    # 模板
    answer_template = """Answer the question {question} using this context: {context}"""
    answer_instructions = answer_template.format(question=question, 
                                                       context=context)    
    
    # 回答
    answer = llm.invoke([SystemMessage(content=answer_instructions)]+[HumanMessage(content=f"Answer the question.")])
      
    # 追加到状态
    return {"answer": answer}

# 添加节点
builder = StateGraph(State)

# 使用 node_secret 初始化每个节点
builder.add_node("search_web",search_web)
builder.add_node("search_wikipedia", search_wikipedia)
builder.add_node("generate_answer", generate_answer)

# 流程
builder.add_edge(START, "search_wikipedia")
builder.add_edge(START, "search_web")
builder.add_edge("search_wikipedia", "generate_answer")
builder.add_edge("search_web", "generate_answer")
builder.add_edge("generate_answer", END)
graph = builder.compile()

display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
result = graph.invoke({"question": "How were Nvidia's Q2 2025 earnings"})
result['answer'].content

### 通过 LangGraph API 使用

**⚠️ 注意**

自录制这些视频以来，我们已更新了 Studio，现在可以在本地运行并通过浏览器访问。这是运行 Studio 的首选方式，而非视频中展示的桌面应用。它现在名为 _LangSmith Studio_（而非 _LangGraph Studio_）。详细设置说明请参见课程开头的"环境搭建"指南。你可以在[此处](https://docs.langchain.com/langsmith/studio)找到 Studio 的说明，在[此处](https://docs.langchain.com/langsmith/quick-start-studio#local-development-server)找到本地部署的具体细节。  
要启动本地开发服务器，请在本模块的 `/studio` 目录下的终端中运行以下命令：

```
langgraph dev
```

你应该看到以下输出：
```
- 🚀 API: http://127.0.0.1:2024
- 🎨 Studio UI: https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024
- 📚 API Docs: http://127.0.0.1:2024/docs
```

打开浏览器并导航到上面显示的 **Studio UI** URL。

In [ ]:
if 'google.colab' in str(get_ipython()):
    raise Exception("Unfortunately LangSmith Studio is currently not supported on Google Colab")

In [ ]:
from langgraph_sdk import get_client
client = get_client(url="http://127.0.0.1:2024")

In [ ]:
thread = await client.threads.create()
input_question = {"question": "How were Nvidia Q2 2025 earnings?"}
async for event in client.runs.stream(thread["thread_id"], 
                                      assistant_id="parallelization", 
                                      input=input_question, 
                                      stream_mode="values"):
    # 检查答案是否已添加到状态
    if event.data is not None:
        answer = event.data.get('answer', None)
        if answer:
            print(answer['content'])